# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syeda-ujala-haider/FlyRANK-Machine-Learning-First-Assignment/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
method = """
MY LANE: Refresh/Content Opportunity Scoring

WHICH MODEL: RANDOM FOREST (primary) + LOGISTIC REGRESSION (comparison)

WHY RANDOM FOREST:
──────────────────
1. My signals are validated (Week ML-06)
   → Non-linear combinations matter

2. Features are clean (Week 3)
   → Won't overfit on garbage

3. Need to explain rankings (production)
   → RF permutation importance tells us which signals matter

4. Baseline is linear scoring
   → RF will show if interactions improve accuracy

WHY LOGISTIC REGRESSION (secondary):
─────────────────────────────────────
1. Simple baseline ML model
2. Fast training + interpretable
3. If LogReg beats Random Forest? Then linear was good enough
4. Compare: Does complexity help?

SUCCESS METRIC:
───────────────
Precision@50: Of top 50 articles ranked by model, how many are actually in top 50?
Precision@20: Of top 20 articles, how many are actually in top 20?

WIN CONDITION:
──────────────
Model beats Week 4 baseline on BOTH metrics
If wins @50 but loses @20: Report both (shows model strength/weakness)
"""

print(method)


MY LANE: Refresh/Content Opportunity Scoring

WHICH MODEL: RANDOM FOREST (primary) + LOGISTIC REGRESSION (comparison)

WHY RANDOM FOREST:
──────────────────
1. My signals are validated (Week ML-06)
   → Non-linear combinations matter
   
2. Features are clean (Week 3)
   → Won't overfit on garbage
   
3. Need to explain rankings (production)
   → RF permutation importance tells us which signals matter
   
4. Baseline is linear scoring
   → RF will show if interactions improve accuracy

WHY LOGISTIC REGRESSION (secondary):
─────────────────────────────────────
1. Simple baseline ML model
2. Fast training + interpretable
3. If LogReg beats Random Forest? Then linear was good enough
4. Compare: Does complexity help?

SUCCESS METRIC:
───────────────
Precision@50: Of top 50 articles ranked by model, how many are actually in top 50?
Precision@20: Of top 20 articles, how many are actually in top 20?

WIN CONDITION:
──────────────
Model beats Week 4 baseline on BOTH metrics
If wins @50 but lo

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
print("="*80)
print("SECTION 2: SPLIT DESIGN")
print("="*80)

split_design = """
CHALLENGE:
My baseline ranked ALL articles (439K rows)
But I manually reviewed only TOP 20
→ I have labels for top 20 only

HONEST SPLIT DESIGN:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Option A: TIME-AWARE SPLIT (Preferred)
─────────────────────────────────────
Use June data:
- Train: June 1-20 (20 days)
- Test: June 21-30 (10 days)

Why:
✅ Time-aware (June data sequentially splits)
✅ Realistic (train on past, test on future)
✅ Same grain as baseline
✅ No data leakage

Option B: GROUPED SPLIT (Also Good)
────────────────────────────────────
Group by client:
- Train: 80% clients (random)
- Test: 20% clients (holdout)

Why:
✅ Grouped (articles from same client in one set)
✅ Realistic (unknown client appears in future)
✅ Prevents overfitting to specific clients

OUR CHOICE: TIME-AWARE (June 1-20 vs June 21-30)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Why this split is honest:
1. Train on past → Test on future (realistic)
2. Both have full feature set (no missing data)
3. Baseline also uses June data (fair comparison)
4. Can measure ranking quality (Precision@50)
"""

print(split_design)

print("\n" + "="*80)
print("IMPLEMENTING THE SPLIT")
print("="*80)

# Code comes next

SECTION 2: SPLIT DESIGN

CHALLENGE:
My baseline ranked ALL articles (439K rows)
But I manually reviewed only TOP 20
→ I have labels for top 20 only

HONEST SPLIT DESIGN:
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Option A: TIME-AWARE SPLIT (Preferred)
─────────────────────────────────────
Use June data:
- Train: June 1-20 (20 days)
- Test: June 21-30 (10 days)

Why:
✅ Time-aware (June data sequentially splits)
✅ Realistic (train on past, test on future)
✅ Same grain as baseline
✅ No data leakage

Option B: GROUPED SPLIT (Also Good)
────────────────────────────────────
Group by client:
- Train: 80% clients (random)
- Test: 20% clients (holdout)

Why:
✅ Grouped (articles from same client in one set)
✅ Realistic (unknown client appears in future)
✅ Prevents overfitting to specific clients

OUR CHOICE: TIME-AWARE (June 1-20 vs June 21-30)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Why this split is honest:
1. Train on past → Test on future (realisti

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score
import warnings
warnings.filterwarnings('ignore')

print("Loading data (sampled)...")

from google.colab import userdata
from huggingface_hub import login
from datasets import load_dataset

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)

dataset = load_dataset("FlyRank/internship-warehouse",
                       data_files="fact_content_daily_performance_sample.parquet")
df = dataset['train'].to_pandas()

# Filter
df = df[df['month'] == '2026-06'].copy()
df = df[(df['gsc_data_available'] == True) &
        (df['ga4_data_available'] == True) &
        (df['gsc_impressions'] >= 10)].drop_duplicates()

# SAMPLE TO REDUCE MEMORY
print(f"Full data: {len(df)} rows")
df = df.sample(n=50000, random_state=42)  # Sample 50K rows only
print(f"Sampled to: {len(df)} rows")

# ============================================================
# CREATE FEATURES
# ============================================================

print("Creating features...")

# CTR Gap
def get_ctr(pos):
    p = int(pos)
    if p <= 1: return 0.32
    elif p >= 10: return 0.05
    else: return {2:0.26, 3:0.20, 4:0.15, 5:0.12, 6:0.10, 7:0.08, 8:0.07}.get(p, 0.10)

df['ctr_expected'] = df['gsc_avg_position'].apply(get_ctr)
df['ctr_actual'] = df['gsc_clicks'] / (df['gsc_impressions'] + 1)
df['ctr_gap'] = df['ctr_expected'] - df['ctr_actual']

# Engagement
df['engagement_rate'] = df['ga4_engaged_sessions'] / (df['ga4_sessions'] + 1)
df['time_on_page'] = df['ga4_total_engagement_sec'] / (df['ga4_sessions'] + 1)

# Log impressions
df['log_imp'] = np.log1p(df['gsc_impressions'])

# AI traffic
df['ai_pct'] = (df['sessions_ai'] / (df['sessions_organic'] + df['sessions_direct'] + 1)) * 100
df['ai_pct'] = df['ai_pct'].clip(0, 100)

# Day
df['day'] = pd.to_datetime(df['report_date']).dt.day

print("✅ Features created")

# ============================================================
# TARGET
# ============================================================

print("Creating target...")

df['baseline_score'] = np.log1p(df['gsc_impressions']) * df['ctr_gap']
top_50 = set(df.nlargest(50, 'baseline_score')['content_hash_id'].unique())
df['is_top_50'] = df['content_hash_id'].isin(top_50).astype(int)

print(f"✅ Target: {df['is_top_50'].sum()} in top 50")

# ============================================================
# SPLIT
# ============================================================

print("Splitting...")

train = df[df['day'] <= 20].copy()
test = df[df['day'] > 20].copy()

print(f"Train: {len(train)} rows")
print(f"Test: {len(test)} rows")

# ============================================================
# PREPARE FEATURES
# ============================================================

features = ['ctr_gap', 'engagement_rate', 'time_on_page', 'log_imp', 'ai_pct']

X_train = train[features].fillna(0)
y_train = train['is_top_50']

X_test = test[features].fillna(0)
y_test = test['is_top_50']

# Scale
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\n✅ READY TO TRAIN")
print(f"Train: {X_train_scaled.shape}")
print(f"Test: {X_test_scaled.shape}")

Loading data (sampled)...
Full data: 439193 rows
Sampled to: 50000 rows
Creating features...
✅ Features created
Creating target...
✅ Target: 84 in top 50
Splitting...
Train: 34807 rows
Test: 15193 rows

✅ READY TO TRAIN
Train: (34807, 5)
Test: (15193, 5)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.